# Advanced Time Series Analysis & Electricity Demand Forecasting

This notebook applies **Seasonal Decomposition**, **Augmented Dickey-Fuller (ADF) Stationarity Testing**, **Holt-Winters Exponential Smoothing**, and **SARIMA** to 36 months of electricity demand data (`2023–2025`).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.metrics import mean_absolute_error, mean_squared_error

df = pd.read_csv('../data/monthly_energy_demand.csv', parse_dates=['Date'], index_col='Date')
df = df.asfreq('MS')
df.head()

In [ ]:
# 1. Seasonal Decomposition (Additive, 12-month period)
decomp = seasonal_decompose(df['Demand_MWh'], model='additive', period=12)
fig = decomp.plot()
fig.set_size_inches(10, 7)
plt.tight_layout()
plt.show()

# 2. Augmented Dickey-Fuller Stationarity Test
adf_raw = adfuller(df['Demand_MWh'])
adf_diff = adfuller(df['Demand_MWh'].diff().dropna())
print(f'Raw Series ADF p-value: {adf_raw[1]:.4f}')
print(f'First-Differenced ADF p-value: {adf_diff[1]:.4f}')

In [ ]:
# 3. Train-Test Split (Train: Jan 2023 - Jun 2025 | Test: Jul 2025 - Dec 2025)
train = df['Demand_MWh'].iloc[:-6]
test = df['Demand_MWh'].iloc[-6:]

# Baseline: 3-Month Moving Average
ma_pred = pd.Series([train.iloc[-3:].mean()] * len(test), index=test.index)

# Holt-Winters Exponential Smoothing
hw_model = ExponentialSmoothing(train, trend='add', seasonal='add', seasonal_periods=12).fit()
hw_pred = hw_model.forecast(len(test))

# SARIMA(1,1,1)(1,1,0,12)
sarima_model = SARIMAX(train, order=(1, 1, 1), seasonal_order=(1, 1, 0, 12)).fit(disp=False)
sarima_pred = sarima_model.forecast(len(test))

def eval_metrics(y_true, y_hat, label):
    mae = mean_absolute_error(y_true, y_hat)
    rmse = np.sqrt(mean_squared_error(y_true, y_hat))
    mape = np.mean(np.abs((y_true - y_hat) / y_true)) * 100
    return {'Model': label, 'MAE (MWh)': round(mae, 1), 'RMSE (MWh)': round(rmse, 1), 'MAPE (%)': round(mape, 2)}

metrics_df = pd.DataFrame([
    eval_metrics(test, ma_pred, '3-Month Moving Average'),
    eval_metrics(test, hw_pred, 'Holt-Winters Seasonal'),
    eval_metrics(test, sarima_pred, 'SARIMA (1,1,1)(1,1,0,12)')
])
print(metrics_df.to_string(index=False))

In [ ]:
# 4. Plot Test Comparison & 6-Month 2026 Forecast
full_hw = ExponentialSmoothing(df['Demand_MWh'], trend='add', seasonal='add', seasonal_periods=12).fit()
future_2026 = full_hw.forecast(6)

plt.figure(figsize=(10, 5))
plt.plot(df.index, df['Demand_MWh'], label='Historical Demand (2023-2025)', color='#1e3a5f', linewidth=2)
plt.plot(future_2026.index, future_2026, label='2026 Forecast (Holt-Winters)', color='#d97706', linestyle='--', marker='o')
plt.title('Monthly Electricity Demand & 6-Month Forecast')
plt.ylabel('Demand (MWh)')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()